# 📅 Day 7 of 100 — Data Cleaning End-to-End

📂 **GitHub Repo:** [100-Days-of-Data-Science / Day 008](https://github.com/imshoaibaslam/100-Days-of-Data-Science/tree/main/Day-08-Datascience)

---

## 🗂️ Agenda
1. Load & Inspect the Raw Dataset
2. Fix Column Names
3. Fix Data Types
4. Handle Missing Values
5. Remove Duplicates
6. Fix Inconsistent String Values
7. Final Clean DataFrame

---

## 🎯 Deliverables
- [ ] Inspect the raw dataset for all issues
- [ ] Rename messy column names
- [ ] Convert columns to correct data types
- [ ] Handle missing values with the right strategy per column
- [ ] Remove duplicate rows
- [ ] Standardize inconsistent string values
- [ ] Produce a final clean DataFrame ready for analysis

---

## 📚 Learning Topics

| # | Topic | Description |
|---|-------|-------------|
| 1 | `df.info()` + `df.describe()` | First pass inspection of the dataset |
| 2 | `rename()` | Fix inconsistent or unclear column names |
| 3 | `astype()` | Convert columns to correct data types |
| 4 | `isnull().sum()` + `fillna()` + `dropna()` | Detect and handle missing values |
| 5 | `duplicated()` + `drop_duplicates()` | Find and remove duplicate rows |
| 6 | `str.strip()` + `str.lower()` + `str.replace()` | Standardize messy string values |





## 🔧 Imports



In [1]:
import pandas as pd
import numpy as np

---

## Step 0 — Load the Dataset

We will use the Wine Reviews dataset — the same one we have been working with throughout this series.
This time we treat it as a **raw, uncleaned dataset** and apply a full cleaning pipeline from scratch.

In [2]:
# Load raw dataset
df = pd.read_csv(r"C:\Users\aslam\Downloads\winemag-data-130k-v2.csv\winemag-data-130k-v2.csv", index_col=0)
df.head(3)

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm


---

## Step 1 — Inspect the Raw Dataset

Before touching anything, we run a **full first-pass inspection**.
Every data cleaning project starts here — you need to know what you are dealing with.

Ask these questions:
- How many rows and columns?
- What are the data types?
- Which columns have missing values — and how many?
- Are there any obvious issues?

In [3]:
# Shape — rows and columns
df.shape

(129971, 13)

In [4]:
# Data types and non-null counts for every column
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 129971 entries, 0 to 129970
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   country                129908 non-null  object 
 1   description            129971 non-null  object 
 2   designation            92506 non-null   object 
 3   points                 129971 non-null  int64  
 4   price                  120975 non-null  float64
 5   province               129908 non-null  object 
 6   region_1               108724 non-null  object 
 7   region_2               50511 non-null   object 
 8   taster_name            103727 non-null  object 
 9   taster_twitter_handle  98758 non-null   object 
 10  title                  129971 non-null  object 
 11  variety                129970 non-null  object 
 12  winery                 129971 non-null  object 
dtypes: float64(1), int64(1), object(11)
memory usage: 13.9+ MB


In [5]:
# Missing values per column — sorted descending
df.isnull().sum().sort_values(ascending=False)

region_2                 79460
designation              37465
taster_twitter_handle    31213
taster_name              26244
region_1                 21247
price                     8996
country                     63
province                    63
variety                      1
description                  0
points                       0
title                        0
winery                       0
dtype: int64

In [6]:
# Statistical summary of numeric columns
df.describe()

,points,price
count,129971.000000,120975.000000
mean,88.447138,35.363389
std,3.039730,41.022218
min,80.000000,4.000000
25%,86.000000,17.000000
50%,88.000000,25.000000
75%,91.000000,42.000000
max,100.000000,3300.000000


**What we found:**

- 129,971 rows × 13 columns
- `region_2` → 79,460 missing (61% of dataset)
- `designation` → 37,465 missing
- `price` → 8,996 missing
- `taster_name` → 26,244 missing
- `country` and `province` → 63 missing each
- `points` stored as `int64` — clean
- `price` stored as `float64` — clean
- All text columns stored as `object` — expected

---

## Step 2 — Fix Column Names

Consistent, readable column names make the rest of the pipeline easier.
Best practice: **lowercase, no spaces, use underscores**.

In [7]:
# Rename columns to more meaningful names
df = df.rename(columns={
    'taster_name'           : 'reviewer',
    'taster_twitter_handle' : 'reviewer_twitter',
    'points'                : 'score',
    'title'                 : 'wine_title'
})

df.columns

Index(['country', 'description', 'designation', 'score', 'price', 'province',
       'region_1', 'region_2', 'reviewer', 'reviewer_twitter', 'wine_title',
       'variety', 'winery'],
      dtype='object')

---

## Step 3 — Fix Data Types

Wrong data types cause silent failures downstream.
Check every column — convert anything that does not match what the data actually represents.

In [8]:
# score is int64 — convert to float64 for consistency with price
df['score'] = df['score'].astype('float64')

# Confirm the change
df[['score', 'price']].dtypes

score    float64
price    float64
dtype: object

---

## Step 4 — Handle Missing Values

### Strategy per column:

| Column | Missing | Strategy | Reason |
|--------|---------|----------|--------|
| `region_2` | 79,460 | Fill with `"Unknown"` | Too many to drop — label as unknown |
| `designation` | 37,465 | Fill with `"Unknown"` | Not critical for analysis |
| `taster_name` | 26,244 | Fill with `"Unknown"` | Reviewer info missing |
| `price` | 8,996 | Fill with median | Numeric — use median to avoid outlier skew |
| `country` | 63 | Drop rows | Small number — dropping is safe |

⚠️ **Rule of thumb:**
- Text columns with missing → fill with `"Unknown"`
- Numeric columns with missing → fill with mean or median
- Very few missing rows → drop them

In [9]:
# Fill text columns with Unknown
df['region_2']    = df['region_2'].fillna('Unknown')
df['designation'] = df['designation'].fillna('Unknown')
df['reviewer']    = df['reviewer'].fillna('Unknown')

In [10]:
# Fill price with median — more robust than mean when outliers exist
df['price'] = df['price'].fillna(df['price'].median())

In [11]:
# Drop rows where country is missing — only 63 rows
df = df.dropna(subset=['country'])

In [12]:
# Confirm — no missing values remaining
df.isnull().sum().sort_values(ascending=False)

reviewer_twitter    31213
region_1            21184
variety                 1
country                 0
description             0
designation             0
score                   0
price                   0
province                0
region_2                0
reviewer                0
wine_title              0
winery                  0
dtype: int64

---

## Step 5 — Remove Duplicates

### What are duplicates?

Duplicate rows appear when:
- Data was loaded from multiple sources
- An API returned the same record twice
- A CSV was accidentally appended to itself

They skew every aggregation — counts, averages, groupby results — silently.

In [13]:
# How many duplicate rows exist?
df.duplicated().sum()

np.int64(9979)

In [14]:
# Remove all duplicate rows — keep first occurrence
df = df.drop_duplicates()

# Confirm
print("Rows after deduplication:", df.shape[0])

Rows after deduplication: 119929


---

## Step 6 — Fix Inconsistent String Values

String columns often have:
- Extra whitespace — `" Italy "` instead of `"Italy"`
- Mixed casing — `"italy"`, `"ITALY"`, `"Italy"` all treated as different
- Inconsistent labels — `"US"` vs `"United States"`

These cause silent groupby errors — the same country appearing as two separate groups.

In [15]:
# Strip leading/trailing whitespace from all string columns
df['country']  = df['country'].str.strip()
df['province'] = df['province'].str.strip()
df['variety']  = df['variety'].str.strip()

In [16]:
# Standardize variety column to title case
df['variety'] = df['variety'].str.title()

# Quick check
df['variety'].value_counts().head(10)

variety
Pinot Noir                  12275
Chardonnay                  10865
Cabernet Sauvignon           8838
Red Blend                    8233
Bordeaux-Style Red Blend     6471
Riesling                     4772
Sauvignon Blanc              4571
Syrah                        3828
Rosé                         3219
Merlot                       2895
Name: count, dtype: int64

In [17]:
# Fix known inconsistent country label
df['country'] = df['country'].replace('US', 'United States')

# Confirm
df['country'].value_counts().head(10)

country
United States    50457
France           20353
Italy            17940
Spain             6116
Portugal          5256
Chile             4184
Argentina         3544
Austria           3034
Australia         2197
Germany           1992
Name: count, dtype: int64

---

## Step 7 — Final Clean DataFrame

In [18]:
# Final inspection
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 119929 entries, 0 to 129970
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   country           119929 non-null  object 
 1   description       119929 non-null  object 
 2   designation       119929 non-null  object 
 3   score             119929 non-null  float64
 4   price             119929 non-null  float64
 5   province          119929 non-null  object 
 6   region_1          100428 non-null  object 
 7   region_2          119929 non-null  object 
 8   reviewer          119929 non-null  object 
 9   reviewer_twitter  90483 non-null   object 
 10  wine_title        119929 non-null  object 
 11  variety           119928 non-null  object 
 12  winery            119929 non-null  object 
dtypes: float64(2), object(11)
memory usage: 12.8+ MB


In [19]:
# Final shape
print("Final dataset:", df.shape)
df.head(5)

Final dataset: (119929, 13)


,country,description,designation,score,price,province,region_1,region_2,reviewer,reviewer_twitter,wine_title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87.0,25.0,Sicily & Sardinia,Etna,Unknown,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87.0,15.0,Douro,NaN,Unknown,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,United States,"Tart and snappy, the flavors of lime flesh and...",Unknown,87.0,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,United States,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87.0,13.0,Michigan,Lake Michigan Shore,Unknown,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,United States,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87.0,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


---

## 💡 The Full Cleaning Checklist

| Step | Action | Tool |
|------|--------|------|
| 1 | Inspect shape, types, nulls | `info()`, `describe()`, `isnull().sum()` |
| 2 | Fix column names | `rename()` |
| 3 | Fix data types | `astype()` |
| 4 | Handle missing values | `fillna()`, `dropna()` |
| 5 | Remove duplicates | `drop_duplicates()` |
| 6 | Fix string inconsistencies | `str.strip()`, `str.title()`, `replace()` |

---

## 🌍 Real World Usage

**This exact pipeline** is what data scientists run on every new dataset before any analysis or modelling begins. In production environments it is often written as a reusable function or a dedicated cleaning script that runs automatically when new data arrives.

Missing values, wrong types, and inconsistent strings are the three most common causes of broken ML pipelines and incorrect business reports. Fixing them upfront saves hours of debugging later.

---

## 🔑 Key Takeaways

- Always inspect before touching — `info()`, `isnull().sum()`, `describe()`
- Different missing value strategies for different columns — one size does not fit all
- `fillna()` fills gaps — `dropna()` removes rows — choose based on how many are missing
- Duplicates skew every aggregation — always check with `duplicated().sum()`
- String inconsistencies cause silent groupby errors — strip, standardize, replace
- A clean DataFrame is not the end goal — it is the starting point

---

## 🏁 Conclusion

Today we ran a **complete end-to-end data cleaning pipeline** on 130,000 real wine reviews — applying every technique from the past week in one structured workflow.

This is the most valuable skill in practical data science. Raw data is always messy. The ability to systematically identify, diagnose, and fix data quality issues is what separates analysts who produce reliable results from those who produce misleading ones.

From Day 10 onwards, we move into Exploratory Data Analysis — using this clean dataset to extract real insights.

---

## 📝 Personal Notes & Observations

> Write anything you learned, got confused about, or want to remember here.

---

## ✅ Day 7 Complete — 93 days to go!

<p align="center">
  <a href="https://github.com/imshoaibaslam/100-Days-of-Data-Science/tree/main/Day-08-Datascience">📂 View Full Code on GitHub</a>
</p>